<a href="https://colab.research.google.com/github/nakthecoder/Online-Courses-Learning/blob/master/Prompt%20Engineering/Cardealership.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#!/usr/bin/env python3
"""
Car Dealership Customer Service Workflow (Enhanced + Trade-In Appraisal)

Adds:
- Test-drive appointment scheduling (multi-turn, with conflict checks).
- Instant trade-in appraisal (multi-turn, collects vehicle details and estimates value range).
Still supports:
- Inventory, finance/lease, location.
- Satisfaction follow-up after each main intent.

No external dependencies.
"""

from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple
from datetime import datetime, timedelta
import re

# ----- Configurable Data ------------------------------------------------------

DEALERSHIP_NAME = "Riverbend Auto"
DEALERSHIP_ADDRESS = "1234 Main St, Springfield, CA 94000"
DEALERSHIP_PHONE = "(555) 123-9876"
DEALERSHIP_HOURS = "Mon–Sat 9:00am–7:00pm, Sun 10:00am–5:00pm"

# Example inventory: model -> list of trims in stock (and colors)
INVENTORY = {
    "Civic": [
        {"trim": "LX", "color": "White", "stock_id": "CIV-LX-WH-1042"},
        {"trim": "EX", "color": "Blue", "stock_id": "CIV-EX-BL-2031"},
    ],
    "Accord": [
        {"trim": "Sport", "color": "Black", "stock_id": "ACC-SP-BK-8854"},
    ],
    "CR-V": [
        {"trim": "EX-L", "color": "Silver", "stock_id": "CRV-EXL-SV-5512"},
        {"trim": "Touring", "color": "Red", "stock_id": "CRV-TO-RD-7789"},
    ],
}

FINANCE_COPY = (
    "We offer flexible finance and lease options:\n"
    "• Financing: Competitive APR for qualified buyers, 24–72 month terms.\n"
    "• Leasing: 24–48 month terms with mileage options and low upfront costs.\n"
    "• Trade-ins: Apply your trade-in value to reduce your payments.\n\n"
    f"For personalized quotes, I can connect you to a finance specialist at {DEALERSHIP_PHONE}."
)

APPOINTMENT_DURATION_MIN = 60  # minutes

# ----- Simple Appointment Store ----------------------------------------------

@dataclass
class Appointment:
    model: str
    when: datetime
    name: Optional[str] = None
    phone: Optional[str] = None

class AppointmentStore:
    def __init__(self):
        self._appointments: List[Appointment] = []

    def is_conflict(self, when: datetime) -> bool:
        for appt in self._appointments:
            delta = abs((appt.when - when).total_seconds()) / 60.0
            if delta < APPOINTMENT_DURATION_MIN:
                return True
        return False

    def add(self, model: str, when: datetime, name: Optional[str], phone: Optional[str]) -> Appointment:
        appt = Appointment(model=model, when=when, name=name, phone=phone)
        self._appointments.append(appt)
        return appt

    def list_for_day(self, day: datetime) -> List[Appointment]:
        return [a for a in self._appointments if a.when.date() == day.date()]

# ----- Trade-In Appraisal -----------------------------------------------------

# Helpful normalizations for makes (not exhaustive, just common examples).
KNOWN_MAKES = {
    "honda", "toyota", "ford", "chevrolet", "chevy", "nissan", "hyundai",
    "kia", "subaru", "mazda", "volkswagen", "vw", "bmw", "mercedes", "mercedes-benz",
    "audi", "lexus", "acura", "infiniti", "volvo", "jeep", "ram", "gmc", "dodge",
    "tesla"
}

CONDITION_LEVELS = ["excellent", "good", "fair", "poor"]

@dataclass
class TradeInDraft:
    year: Optional[int] = None
    make: Optional[str] = None
    model: Optional[str] = None
    mileage: Optional[int] = None
    condition: Optional[str] = None  # one of CONDITION_LEVELS
    zip_code: Optional[str] = None
    vin: Optional[str] = None
    accidents: Optional[bool] = None
    name: Optional[str] = None
    phone: Optional[str] = None

@dataclass
class TradeInEstimate:
    low: int
    high: int
    assumptions: str

class TradeInAppraiser:
    """
    Very simple, transparent heuristic for ballpark trade-in values.
    Not a binding offer. You can replace this with a real pricing engine/API.
    """
    def estimate(self, *, year: int, make: str, model: str,
                 mileage: int, condition: str, accidents: Optional[bool]) -> TradeInEstimate:
        now_year = datetime.now().year
        age = max(0, now_year - year)
        make_norm = make.lower()

        # Baseline by brand segment (very rough)
        segment_baseline = 35000
        if make_norm in {"bmw", "mercedes", "mercedes-benz", "audi", "lexus", "infiniti", "tesla", "volvo"}:
            segment_baseline = 45000
        elif make_norm in {"kia", "hyundai", "chevrolet", "chevy", "ford", "nissan", "volkswagen", "vw", "subaru"}:
            segment_baseline = 32000
        elif make_norm in {"jeep", "gmc", "ram", "dodge"}:
            segment_baseline = 38000

        # Depreciation: 18% first year, 12% each additional (compounded)
        value = float(segment_baseline)
        if age >= 1:
            value *= (1 - 0.18)
            for _ in range(age - 1):
                value *= (1 - 0.12)

        # Mileage adjustment vs. expected (12k per year)
        expected_miles = 12000 * age
        delta = mileage - expected_miles
        per_mile_penalty = 0.10  # $/mile over expected
        per_mile_credit = 0.05   # $/mile under expected
        if delta > 0:
            value -= per_mile_penalty * min(delta, 200000)  # cap
        else:
            value += per_mile_credit * min(abs(delta), 100000)

        # Condition multiplier
        cond = (condition or "good").lower()
        cond_mult = {"excellent": 1.05, "good": 1.00, "fair": 0.90, "poor": 0.75}.get(cond, 1.0)
        value *= cond_mult

        # Accident history
        if accidents is True:
            value *= 0.90  # -10% if accident(s) reported

        # Floor/ceiling bounds
        value = max(500.0, value)
        low = int(max(500, round(value * 0.92)))
        high = int(round(value * 1.08))

        assumptions = (
            f"Assumptions: age={age}y, expected_miles≈{expected_miles:,}, "
            f"condition='{cond}', accidents={'yes' if accidents else 'no/unknown'}."
        )
        return TradeInEstimate(low=low, high=high, assumptions=assumptions)

# ----- Intent Detection -------------------------------------------------------

INTENT_KEYWORDS = {
    "inventory": [
        "available", "availability", "in stock", "stock", "do you have",
        "inventory", "on the lot", "units", "color", "trim", "model",
        "civic", "accord", "cr-v", "crv"
    ],
    "finance": [
        "finance", "financing", "lease", "leasing", "apr", "monthly",
        "payment", "payments", "credit", "terms", "quote"
    ],
    "location": [
        "where", "address", "location", "hours", "open", "close", "directions",
        "nearby", "phone", "contact"
    ],
    "appointment": [
        "test drive", "test-drive", "schedule", "appointment", "book", "book a drive",
        "set up a drive", "come in", "schedule a time", "testdriv"
    ],
    "tradein": [
        "trade in", "trade-in", "tradein", "sell my car", "value my car",
        "appraisal", "how much is my car worth", "kbb", "offer for my car"
    ],
    "affirmation": ["yes", "yep", "yeah", "sure", "that helps", "thanks", "good"],
    "negation": ["no", "nope", "not really", "didn't", "doesn't", "still", "confused"],
}

def detect_intent(text: str) -> Optional[str]:
    t = text.lower()

    # Satisfaction answers (priority when waiting)
    for word in INTENT_KEYWORDS["affirmation"]:
        if re.search(rf"\b{re.escape(word)}\b", t):
            return "affirmation"
    for word in INTENT_KEYWORDS["negation"]:
        if re.search(rf"\b{re.escape(word)}\b", t):
            return "negation"

    hits = []
    for intent in ("tradein", "appointment", "inventory", "finance", "location"):
        score = sum(1 for w in INTENT_KEYWORDS[intent] if w in t)
        if score:
            hits.append((intent, score))
    if hits:
        hits.sort(key=lambda x: x[1], reverse=True)
        return hits[0][0]
    return None

# ----- Helpers: model + datetime parsing -------------------------------------

def extract_models(text: str) -> List[str]:
    t = text.lower()
    return [m for m in INVENTORY if m.lower() in t or m.lower().replace("-", "") in t]

_TIME_PATTERNS = [
    "%Y-%m-%d %H:%M",
    "%Y-%m-%d %I:%M %p",
    "%m/%d/%Y %H:%M",
    "%m/%d/%Y %I:%M %p",
    "%m/%d %H:%M",      # assume current year
    "%m/%d %I:%M %p",   # assume current year
    "%Y-%m-%d %H%M",
]

def _normalize_year(dt_like: datetime) -> datetime:
    now = datetime.now()
    return dt_like.replace(year=now.year)

def parse_datetime_natural(text: str) -> Optional[datetime]:
    t = text.strip().lower()

    def parse_time_fragment(fragment: str) -> Optional[Tuple[int, int]]:
        fragment = fragment.strip()
        m = re.match(r"^(\d{1,2})(?::(\d{2}))?\s*(am|pm)$", fragment)
        if m:
            h = int(m.group(1)); mnt = int(m.group(2) or 0); ampm = m.group(3)
            if ampm == "pm" and h < 12: h += 12
            if ampm == "am" and h == 12: h = 0
            return h, mnt
        m = re.match(r"^(\d{1,2}):(\d{2})$", fragment)
        if m:
            return int(m.group(1)), int(m.group(2))
        return None

    if "tomorrow" in t or "today" in t:
        base = datetime.now()
        if "tomorrow" in t:
            base = base + timedelta(days=1)
        m = re.search(r"(?:today|tomorrow)\s+([\d:]{1,5}\s*(?:am|pm)?|\d{1,2}:\d{2})", t)
        if m:
            timefrag = m.group(1).replace(".", "")
            hm = parse_time_fragment(timefrag)
            if hm:
                h, mnt = hm
                return base.replace(hour=h, minute=mnt, second=0, microsecond=0)

    for fmt in _TIME_PATTERNS:
        try:
            dt = datetime.strptime(t, fmt)
            if "%Y" not in fmt:
                dt = _normalize_year(dt)
            return dt
        except ValueError:
            continue

    m = re.search(
        r"(jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec)[a-z]*\s+(\d{1,2}).*?(\d{1,2})(?::(\d{2}))?\s*(am|pm)",
        t
    )
    if m:
        month_str, day, hour, minute, ampm = m.groups()
        month_map = {"jan":1,"feb":2,"mar":3,"apr":4,"may":5,"jun":6,"jul":7,"aug":8,"sep":9,"oct":10,"nov":11,"dec":12}
        month = month_map[month_str[:3]]
        hour = int(hour); minute = int(minute or 0)
        if ampm == "pm" and hour < 12: hour += 12
        if ampm == "am" and hour == 12: hour = 0
        now = datetime.now()
        return datetime(now.year, month, int(day), hour, minute, 0)

    return None

# ----- Responses: inventory / finance / location -----------------------------

def respond_inventory(user_text: str) -> str:
    t = user_text.lower()
    mentioned_models = [m for m in INVENTORY.keys() if m.lower() in t or m.lower().replace("-", "") in t]
    if mentioned_models:
        parts: List[str] = []
        for model in mentioned_models:
            trims = INVENTORY.get(model, [])
            if trims:
                lines = [f"- {model} in stock:"]
                for item in trims:
                    lines.append(f"  • {item['trim']} – {item['color']} (ID: {item['stock_id']})")
                parts.append("\n".join(lines))
            else:
                parts.append(f"- {model}: currently out of stock.")
        body = "\n".join(parts)
        return (
            f"Here’s what we have right now:\n{body}\n\n"
            "If you want, I can place a hold or text you when new units arrive."
        )
    else:
        summary_lines = [f"- {model}: {len(items)} in stock" for model, items in INVENTORY.items()]
        summary = "\n".join(summary_lines)
        return (
            "We’ve got several popular models available right now:\n"
            f"{summary}\n\n"
            "Tell me which model/trim/color you’re interested in, and I can check exact availability."
        )

def respond_finance(_: str) -> str:
    return FINANCE_COPY

def respond_location(_: str) -> str:
    return (
        f"{DEALERSHIP_NAME}\n"
        f"Address: {DEALERSHIP_ADDRESS}\n"
        f"Phone: {DEALERSHIP_PHONE}\n"
        f"Hours: {DEALERSHIP_HOURS}\n\n"
        "We’re just off the freeway with plenty of parking. Would you like directions?"
    )

# ----- Appointment Orchestrator ----------------------------------------------

@dataclass
class AppointmentDraft:
    model: Optional[str] = None
    when: Optional[datetime] = None
    name: Optional[str] = None
    phone: Optional[str] = None

@dataclass
class ConversationState:
    last_service_intent: Optional[str] = None
    awaiting_satisfaction: bool = False
    history: List[Tuple[str, str]] = field(default_factory=list)
    # Appointment flow
    collecting_appointment: bool = False
    appointment_draft: AppointmentDraft = field(default_factory=AppointmentDraft)
    # Trade-in flow
    collecting_tradein: bool = False
    tradein_draft: TradeInDraft = field(default_factory=TradeInDraft)

class CustomerServiceBot:
    def __init__(self):
        self.state = ConversationState()
        self.appts = AppointmentStore()
        self.appraiser = TradeInAppraiser()

    # ---- Core routing ----
    def handle_message(self, user_text: str) -> str:
        intent = detect_intent(user_text)

        # Active multi-turn flows take precedence (except direct satisfaction replies)
        if self.state.collecting_appointment and intent not in ("affirmation", "negation"):
            reply = self._appointment_collect(user_text)
            self._log(user_text, reply)
            return reply

        if self.state.collecting_tradein and intent not in ("affirmation", "negation"):
            reply = self._tradein_collect(user_text)
            self._log(user_text, reply)
            return reply

        # Satisfaction handling
        if self.state.awaiting_satisfaction and intent in ("affirmation", "negation"):
            self.state.awaiting_satisfaction = False
            if intent == "affirmation":
                reply = "Great! If there’s anything else you need—test drives, quotes, or trade-ins—just let me know."
                self._log(user_text, reply)
                return reply
            else:
                next_step = {
                    "inventory": "Which model/trim should I check specifically? I can also set a stock alert for you.",
                    "finance": "What monthly budget or term are you considering? I can draft a sample quote.",
                    "location": "Are you starting from a specific address? I can send step-by-step directions.",
                    "appointment": "Would you like a morning or afternoon slot? I can suggest a few times.",
                    "tradein": "I can refine your appraisal—do you know the VIN or have photos? We can also book a 15-minute in-person check.",
                }.get(self.state.last_service_intent, "Tell me what you’d like to do next and I’ll take care of it.")
                reply = f"Thanks for the feedback. Let’s get this right. {next_step}"
                self._log(user_text, reply)
                return reply

        # Main intents
        if intent == "inventory":
            answer = respond_inventory(user_text)
            reply = self._with_satisfaction(answer, "inventory")
        elif intent == "finance":
            answer = respond_finance(user_text)
            reply = self._with_satisfaction(answer, "finance")
        elif intent == "location":
            answer = respond_location(user_text)
            reply = self._with_satisfaction(answer, "location")
        elif intent == "appointment":
            reply = self._appointment_start(user_text)
        elif intent == "tradein":
            reply = self._tradein_start(user_text)
        elif intent in ("affirmation", "negation"):
            reply = "Awesome! Anything else I can help with today?" if intent == "affirmation" \
                    else "No problem—what info would you like me to clarify or provide next?"
        else:
            reply = (
                "I can help with inventory availability, finance/lease options, our location/hours, "
                "schedule a test-drive, or give you an instant trade-in appraisal.\n"
                "What would you like to do?"
            )

        self._log(user_text, reply)
        return reply

    # ---- Shared helpers ----
    def _with_satisfaction(self, message: str, intent_name: str) -> str:
        self.state.last_service_intent = intent_name
        self.state.awaiting_satisfaction = True
        return f"{message}\n\nDid that answer your question? (yes/no)"

    def _log(self, user_text: str, bot_text: str) -> None:
        self.state.history.append((user_text, bot_text))

    # ---- Appointment Flow ----
    def _appointment_start(self, user_text: str) -> str:
        self.state.last_service_intent = "appointment"
        self.state.collecting_appointment = True
        models = extract_models(user_text)
        if models:
            self.state.appointment_draft.model = models[0]
        when = parse_datetime_natural(user_text)
        if when:
            self.state.appointment_draft.when = when
        return self._appointment_collect(user_text)

    def _appointment_collect(self, user_text: str) -> str:
        draft = self.state.appointment_draft
        if not draft.model:
            models = extract_models(user_text)
            if models:
                draft.model = models[0]
        if not draft.when:
            pdt = parse_datetime_natural(user_text)
            if pdt:
                draft.when = pdt
        if not draft.name:
            m = re.search(r"\bmy name is ([a-z ,.'-]+)", user_text, re.I)
            if m:
                draft.name = m.group(1).strip().title()
        if not draft.phone:
            m = re.search(r"(\+?1?[-.\s]?)?\(?\d{3}\)?[-.\s]?\d{3}[-.\s]?\d{4}\b", user_text)
            if m:
                draft.phone = m.group(0)

        if not draft.model:
            available = ", ".join(INVENTORY.keys())
            return f"Happy to set that up! Which model would you like to test drive? (e.g., {available})"

        if not draft.when:
            return (
                "Great choice. What date and time works for you? "
                "You can say things like '2025-10-12 14:30', '10/12 2:30 pm', or 'tomorrow 3pm'."
            )

        if not self._is_within_hours(draft.when):
            return (
                "We’re open Mon–Sat 9:00am–7:00pm and Sun 10:00am–5:00pm. "
                "Please pick a time within business hours."
            )

        if self.appts.is_conflict(draft.when):
            alt1 = draft.when + timedelta(minutes=90)
            alt2 = draft.when + timedelta(minutes=150)
            return (
                f"That time is booked. Would {alt1.strftime('%a %b %d, %I:%M %p')} or "
                f"{alt2.strftime('%a %b %d, %I:%M %p')} work instead?"
            )

        appt = self.appts.add(draft.model, draft.when, draft.name, draft.phone)
        self.state.collecting_appointment = False
        self.state.appointment_draft = AppointmentDraft()
        self.state.awaiting_satisfaction = True
        self.state.last_service_intent = "appointment"

        contact_line = ""
        if appt.name or appt.phone:
            pieces = []
            if appt.name: pieces.append(appt.name)
            if appt.phone: pieces.append(appt.phone)
            contact_line = f"\nContact: {', '.join(pieces)}"

        confirmation = (
            f"All set! I’ve scheduled your {appt.model} test drive for "
            f"{appt.when.strftime('%A, %B %d at %I:%M %p')}.{contact_line}\n"
            "Please bring your driver’s license.\n\n"
            "Did that answer your request? (yes/no)"
        )
        return confirmation

    def _is_within_hours(self, when: datetime) -> bool:
        weekday = when.weekday()  # Mon=0, Sun=6
        hour = when.hour + when.minute/60.0
        if weekday == 6:  # Sunday
            return 10 <= hour < 17
        return 9 <= hour < 19

    # ---- Trade-in Flow ----
    def _tradein_start(self, user_text: str) -> str:
        self.state.last_service_intent = "tradein"
        self.state.collecting_tradein = True
        # Try to prefill from free text on entry
        self._tradein_autofill(user_text)
        return self._tradein_collect(user_text)

    def _tradein_autofill(self, text: str) -> None:
        d = self.state.tradein_draft
        t = text.lower()

        if d.year is None:
            m = re.search(r"\b(20\d{2}|19\d{2})\b", t)
            if m:
                d.year = int(m.group(1))
        if d.mileage is None:
            m = re.search(r"\b(\d{1,3}[,\s]?\d{3}|\d{1,6})\s*miles?\b", t)
            if m:
                d.mileage = int(re.sub(r"[^\d]", "", m.group(1)))
        if d.zip_code is None:
            m = re.search(r"\b(\d{5})(?:-\d{4})?\b", t)
            if m:
                d.zip_code = m.group(1)
        if d.vin is None:
            m = re.search(r"\b([0-9a-hj-npr-z]{17})\b", t)  # basic VIN (no I,O,Q)
            if m:
                d.vin = m.group(1).upper()
        if d.accidents is None:
            if "no accident" in t or "clean title" in t:
                d.accidents = False
            elif "accident" in t or "salvage" in t:
                d.accidents = True
        if d.condition is None:
            for lvl in CONDITION_LEVELS:
                if lvl in t:
                    d.condition = lvl
                    break
        if d.make is None or d.model is None:
            tokens = re.findall(r"[a-z0-9\-]+", t)
            # crude: first known make + next token as model
            for i, tok in enumerate(tokens):
                if tok in KNOWN_MAKES and i + 1 < len(tokens):
                    d.make = tokens[i]
                    d.model = tokens[i + 1]
                    break
        if d.name is None:
            m = re.search(r"\bmy name is ([a-z ,.'-]+)", text, re.I)
            if m:
                d.name = m.group(1).strip().title()
        if d.phone is None:
            m = re.search(r"(\+?1?[-.\s]?)?\(?\d{3}\)?[-.\s]?\d{3}[-.\s]?\d{4}\b", text)
            if m:
                d.phone = m.group(0)

    def _tradein_collect(self, user_text: str) -> str:
        d = self.state.tradein_draft
        # Keep trying to learn from free text each turn
        self._tradein_autofill(user_text)

        # Ask for required fields in order
        if d.year is None:
            return "Let’s get your instant appraisal. What **year** is your vehicle? (e.g., 2018)"
        if d.make is None:
            return "Great—what **make** is it? (e.g., Honda, Toyota, Ford)"
        if d.model is None:
            return "And what **model** is it? (e.g., Civic, RAV4, F-150)"
        if d.mileage is None:
            return "Approximately how many **miles** are on it?"
        if d.condition is None:
            return "How would you rate the **condition**? (excellent / good / fair / poor)"
        if d.accidents is None:
            return "Any **accidents** or reported damage? (yes/no)"
        # Optional, but we’ll ask once
        if d.zip_code is None:
            return "What’s your **ZIP code**? (optional, helps us tailor local demand) Or say 'skip'."
        if d.vin is None:
            return "If you have the **VIN**, share it (optional). Or say 'skip'."

        # Compute estimate
        estimate = self.appraiser.estimate(
            year=d.year, make=d.make, model=d.model,
            mileage=d.mileage, condition=d.condition,
            accidents=d.accidents
        )

        self.state.collecting_tradein = False
        self.state.awaiting_satisfaction = True
        self.state.last_service_intent = "tradein"

        opt_contact = []
        if d.name: opt_contact.append(d.name)
        if d.phone: opt_contact.append(d.phone)
        contact_line = f"\nContact: {', '.join(opt_contact)}" if opt_contact else ""

        # Clear draft for next time
        self.state.tradein_draft = TradeInDraft()

        return (
            "Here’s your **instant trade-in estimate** (not a final offer):\n"
            f"• Estimated range: ${estimate.low:,} – ${estimate.high:,}\n"
            f"• Vehicle: {d.year} {d.make.title()} {d.model.title()} | {d.mileage:,} miles | condition: {d.condition}\n"
            f"• {estimate.assumptions}\n"
            f"{contact_line}\n\n"
            "Next steps: we can **lock in a firmer offer** after a quick in-person check (10–15 minutes). "
            "Would you like to book a visit or apply this toward a new vehicle?\n\n"
            "Did that answer your request? (yes/no)"
        )

# ----- CLI Demo ---------------------------------------------------------------

def run_cli_demo():
    print(f"Welcome to {DEALERSHIP_NAME} virtual assistant! Type 'exit' to quit.\n")
    bot = CustomerServiceBot()
    while True:
        try:
            user = input("You: ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\nGoodbye!")
            break
        if user.lower() in ("exit", "quit"):
            print("Bot: Thanks for visiting. Have a great day!")
            break
        response = bot.handle_message(user)
        print(f"Bot: {response}\n")

if __name__ == "__main__":
    run_cli_demo()


Welcome to Riverbend Auto virtual assistant! Type 'exit' to quit.

You: Hello
Bot: I can help with inventory availability, finance/lease options, our location/hours, schedule a test-drive, or give you an instant trade-in appraisal.
What would you like to do?

You: Buy a car
Bot: I can help with inventory availability, finance/lease options, our location/hours, schedule a test-drive, or give you an instant trade-in appraisal.
What would you like to do?

You: i want a porshe 
Bot: I can help with inventory availability, finance/lease options, our location/hours, schedule a test-drive, or give you an instant trade-in appraisal.
What would you like to do?



In [ ]:
#!/usr/bin/env python3
"""
Dealership Assistant — All-in-One

Features (implements “all” add‑ons):
1) Core workflows: inventory, finance/lease, location, test-drive scheduling, instant trade‑in appraisal.
2) Satisfaction follow‑ups after every main intent.
3) Persistence: appointments/appraisals/history saved to JSON; CSV export helpers.
4) Admin commands: list/cancel appointments, list/export history, export appraisals.
5) SMS/Email confirmation stubs (swap in real providers later).
6) Web API (FastAPI) with endpoints for chat, appointments, appraisals, admin ops.
7) Improved NLP: regex + weighted keywords with simple tie‑breaks.
8) Unit tests (unittest) covering intent detection, appointment hours, and appraisal math.

Usage:
- CLI chat (no server):
    python dealership_assistant.py --cli

- Start API server (requires FastAPI + Uvicorn):
    python dealership_assistant.py --api --host 0.0.0.0 --port 8000

- Admin: list/cancel/export
    python dealership_assistant.py --admin list-appointments
    python dealership_assistant.py --admin cancel-appointment --at "2025-10-12 14:30"
    python dealership_assistant.py --admin export-appointments --csv out/appts.csv
    python dealership_assistant.py --admin export-history --csv out/history.csv
    python dealership_assistant.py --admin export-appraisals --csv out/appraisals.csv

- Run unit tests:
    python dealership_assistant.py --test

Persistence files (created automatically):
    data/appointments.json
    data/appraisals.json
    data/history.json

Note: The API layer depends on: fastapi, uvicorn, pydantic (v1 or v2). Install via:
    pip install fastapi uvicorn pydantic
"""
from __future__ import annotations

import argparse
import csv
import json
import os
import re
from dataclasses import dataclass, asdict, field
from datetime import datetime, timedelta
from typing import Dict, List, Optional, Tuple, Any

# ---------- Configurable Dealership Data -------------------------------------

DEALERSHIP_NAME = "Riverbend Auto"
DEALERSHIP_ADDRESS = "1234 Main St, Springfield, CA 94000"
DEALERSHIP_PHONE = "(555) 123-9876"
DEALERSHIP_HOURS = "Mon–Sat 9:00am–7:00pm, Sun 10:00am–5:00pm"
APPOINTMENT_DURATION_MIN = 60  # minutes

INVENTORY: Dict[str, List[Dict[str, str]]] = {
    "Civic": [
        {"trim": "LX", "color": "White", "stock_id": "CIV-LX-WH-1042"},
        {"trim": "EX", "color": "Blue", "stock_id": "CIV-EX-BL-2031"},
    ],
    "Accord": [
        {"trim": "Sport", "color": "Black", "stock_id": "ACC-SP-BK-8854"},
    ],
    "CR-V": [
        {"trim": "EX-L", "color": "Silver", "stock_id": "CRV-EXL-SV-5512"},
        {"trim": "Touring", "color": "Red", "stock_id": "CRV-TO-RD-7789"},
    ],
}

FINANCE_COPY = (
    "We offer flexible finance and lease options:\n"
    "• Financing: Competitive APR for qualified buyers, 24–72 month terms.\n"
    "• Leasing: 24–48 month terms with mileage options and low upfront costs.\n"
    "• Trade-ins: Apply your trade-in value to reduce your payments.\n\n"
    f"For personalized quotes, I can connect you to a finance specialist at {DEALERSHIP_PHONE}."
)

# ---------- Persistence Layer -------------------------------------------------

DATA_DIR = os.path.join(os.getcwd(), "data")
APPTS_PATH = os.path.join(DATA_DIR, "appointments.json")
APPRAISALS_PATH = os.path.join(DATA_DIR, "appraisals.json")
HISTORY_PATH = os.path.join(DATA_DIR, "history.json")

os.makedirs(DATA_DIR, exist_ok=True)


def _load_json(path: str) -> List[Dict[str, Any]]:
    if not os.path.exists(path):
        return []
    try:
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    except json.JSONDecodeError:
        return []


def _save_json(path: str, rows: List[Dict[str, Any]]):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(rows, f, indent=2, ensure_ascii=False, default=str)


# ---------- Domain Models -----------------------------------------------------

@dataclass
class Appointment:
    model: str
    when: str  # ISO string for persistence
    name: Optional[str] = None
    phone: Optional[str] = None

    @property
    def when_dt(self) -> datetime:
        return datetime.fromisoformat(self.when)


@dataclass
class Appraisal:
    created_at: str  # ISO
    year: int
    make: str
    model: str
    mileage: int
    condition: str
    accidents: Optional[bool]
    zip_code: Optional[str] = None
    vin: Optional[str] = None
    name: Optional[str] = None
    phone: Optional[str] = None
    low: int = 0
    high: int = 0
    assumptions: str = ""


# ---------- Stores with Persistence -----------------------------------------

class AppointmentStore:
    def __init__(self, path: str = APPTS_PATH):
        self.path = path
        self._rows: List[Appointment] = [Appointment(**r) for r in _load_json(self.path)]

    def save(self):
        _save_json(self.path, [asdict(a) for a in self._rows])

    def is_conflict(self, when_dt: datetime) -> bool:
        for appt in self._rows:
            delta = abs((appt.when_dt - when_dt).total_seconds()) / 60.0
            if delta < APPOINTMENT_DURATION_MIN:
                return True
        return False

    def add(self, appt: Appointment) -> Appointment:
        self._rows.append(appt)
        self.save()
        return appt

    def list_all(self) -> List[Appointment]:
        return list(self._rows)

    def cancel_at(self, when_iso: str) -> bool:
        before = len(self._rows)
        self._rows = [a for a in self._rows if a.when != when_iso]
        changed = len(self._rows) != before
        if changed:
            self.save()
        return changed

    def export_csv(self, csv_path: str):
        os.makedirs(os.path.dirname(csv_path) or ".", exist_ok=True)
        with open(csv_path, "w", newline="", encoding="utf-8") as f:
            w = csv.DictWriter(f, fieldnames=["model", "when", "name", "phone"])
            w.writeheader()
            for a in self._rows:
                w.writerow(asdict(a))


class HistoryStore:
    def __init__(self, path: str = HISTORY_PATH):
        self.path = path
        self._rows: List[Dict[str, str]] = _load_json(self.path)

    def append(self, user_text: str, bot_text: str):
        self._rows.append({
            "timestamp": datetime.utcnow().isoformat(),
            "user": user_text,
            "bot": bot_text,
        })
        self.save()

    def save(self):
        _save_json(self.path, self._rows)

    def list_all(self) -> List[Dict[str, str]]:
        return list(self._rows)

    def export_csv(self, csv_path: str):
        os.makedirs(os.path.dirname(csv_path) or ".", exist_ok=True)
        with open(csv_path, "w", newline="", encoding="utf-8") as f:
            w = csv.DictWriter(f, fieldnames=["timestamp", "user", "bot"])
            w.writeheader()
            for r in self._rows:
                w.writerow(r)


class AppraisalStore:
    def __init__(self, path: str = APPRAISALS_PATH):
        self.path = path
        self._rows: List[Appraisal] = [Appraisal(**r) for r in _load_json(self.path)]

    def add(self, appr: Appraisal) -> Appraisal:
        self._rows.append(appr)
        self.save()
        return appr

    def save(self):
        _save_json(self.path, [asdict(a) for a in self._rows])

    def list_all(self) -> List[Appraisal]:
        return list(self._rows)

    def export_csv(self, csv_path: str):
        os.makedirs(os.path.dirname(csv_path) or ".", exist_ok=True)
        with open(csv_path, "w", newline="", encoding="utf-8") as f:
            w = csv.DictWriter(f, fieldnames=[
                "created_at","year","make","model","mileage","condition","accidents",
                "zip_code","vin","name","phone","low","high","assumptions"
            ])
            w.writeheader()
            for a in self._rows:
                w.writerow(asdict(a))


# ---------- Utility: Messaging Stubs (SMS/Email) -----------------------------

def send_sms_stub(phone: Optional[str], body: str) -> None:
    """Stub for SMS — integrate Twilio/etc. later."""
    if phone:
        print(f"[SMS → {phone}] {body}")


def send_email_stub(email: Optional[str], subject: str, body: str) -> None:
    """Stub for Email — integrate SES/SendGrid/etc. later."""
    if email:
        print(f"[EMAIL → {email}] {subject}\n{body}")


# ---------- NLP: Improved Intent Detection ----------------------------------

KEYWORDS = {
    "inventory": ["available","availability","in stock","stock","do you have","inventory","on the lot","units","color","trim","model","civic","accord","cr-v","crv"],
    "finance": ["finance","financing","lease","leasing","apr","monthly","payment","payments","credit","terms","quote"],
    "location": ["where","address","location","hours","open","close","directions","nearby","phone","contact"],
    "appointment": ["test drive","test-drive","schedule","appointment","book","book a drive","set up a drive","come in","schedule a time","testdriv"],
    "tradein": ["trade in","trade-in","tradein","sell my car","value my car","appraisal","how much is my car worth","kbb","offer for my car"],
}
AFFIRM = ["yes","yep","yeah","sure","that helps","thanks","good"]
NEGATE = ["no","nope","not really","didn't","doesn't","still","confused"]

# Lightweight pattern boosts (regexes) for precision
PATTERNS = {
    "appointment": [re.compile(r"\b(test\s*-?\s*drive|schedule|appointment|book)\b", re.I)],
    "tradein": [re.compile(r"\b(trade\s*-?\s*in|appraisal|sell my car|value my car)\b", re.I)],
}


def detect_intent(text: str) -> Optional[str]:
    t = text.lower()
    # Satisfaction quick path
    if any(re.search(rf"\b{re.escape(w)}\b", t) for w in AFFIRM):
        return "affirmation"
    if any(re.search(rf"\b{re.escape(w)}\b", t) for w in NEGATE):
        return "negation"

    scores: Dict[str, float] = {}
    for intent, kws in KEYWORDS.items():
        score = 0.0
        for kw in kws:
            if kw in t:
                score += 1.0
        # regex boosts
        for rx in PATTERNS.get(intent, []):
            if rx.search(text):
                score += 1.5
        if score > 0:
            scores[intent] = score

    if not scores:
        return None
    # pick highest, tie-break by custom priority
    priority = ["tradein","appointment","inventory","finance","location"]
    best = max(scores.items(), key=lambda kv: (kv[1], -priority.index(kv[0]) if kv[0] in priority else 0))
    return best[0]


# ---------- Helpers: Parsing --------------------------------------------------

KNOWN_MAKES = {"honda","toyota","ford","chevrolet","chevy","nissan","hyundai","kia","subaru","mazda","volkswagen","vw","bmw","mercedes","mercedes-benz","audi","lexus","acura","infiniti","volvo","jeep","ram","gmc","dodge","tesla"}
CONDITION_LEVELS = ["excellent","good","fair","poor"]


def extract_models_from_text(text: str) -> List[str]:
    t = text.lower()
    return [m for m in INVENTORY if m.lower() in t or m.lower().replace("-", "") in t]


_TIME_PATTERNS = [
    "%Y-%m-%d %H:%M",
    "%Y-%m-%d %I:%M %p",
    "%m/%d/%Y %H:%M",
    "%m/%d/%Y %I:%M %p",
    "%m/%d %H:%M",
    "%m/%d %I:%M %p",
    "%Y-%m-%d %H%M",
]


def _normalize_year(dt_like: datetime) -> datetime:
    now = datetime.now()
    return dt_like.replace(year=now.year)


def parse_datetime_natural(text: str) -> Optional[datetime]:
    t = text.strip().lower()

    def parse_time_fragment(fragment: str) -> Optional[Tuple[int, int]]:
        fragment = fragment.strip()
        m = re.match(r"^(\d{1,2})(?::(\d{2}))?\s*(am|pm)$", fragment)
        if m:
            h = int(m.group(1)); mnt = int(m.group(2) or 0); ampm = m.group(3)
            if ampm == "pm" and h < 12: h += 12
            if ampm == "am" and h == 12: h = 0
            return h, mnt
        m = re.match(r"^(\d{1,2}):(\d{2})$", fragment)
        if m:
            return int(m.group(1)), int(m.group(2))
        return None

    if "tomorrow" in t or "today" in t:
        base = datetime.now()
        if "tomorrow" in t:
            base = base + timedelta(days=1)
        m = re.search(r"(?:today|tomorrow)\s+([\d:]{1,5}\s*(?:am|pm)?|\d{1,2}:\d{2})", t)
        if m:
            timefrag = m.group(1).replace(".", "")
            hm = parse_time_fragment(timefrag)
            if hm:
                h, mnt = hm
                return base.replace(hour=h, minute=mnt, second=0, microsecond=0)

    for fmt in _TIME_PATTERNS:
        try:
            dt = datetime.strptime(t, fmt)
            if "%Y" not in fmt:
                dt = _normalize_year(dt)
            return dt
        except ValueError:
            continue

    m = re.search(r"(jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec)[a-z]*\s+(\d{1,2}).*?(\d{1,2})(?::(\d{2}))?\s*(am|pm)", t)
    if m:
        month_str, day, hour, minute, ampm = m.groups()
        month_map = {"jan":1,"feb":2,"mar":3,"apr":4,"may":5,"jun":6,"jul":7,"aug":8,"sep":9,"oct":10,"nov":11,"dec":12}
        month = month_map[month_str[:3]]
        hour = int(hour); minute = int(minute or 0)
        if ampm == "pm" and hour < 12: hour += 12
        if ampm == "am" and hour == 12: hour = 0
        now = datetime.now()
        return datetime(now.year, month, int(day), hour, minute, 0)

    return None


# ---------- Responses ---------------------------------------------------------

def respond_inventory(user_text: str) -> str:
    t = user_text.lower()
    mentioned_models = [m for m in INVENTORY if m.lower() in t or m.lower().replace("-", "") in t]
    if mentioned_models:
        parts: List[str] = []
        for model in mentioned_models:
            trims = INVENTORY.get(model, [])
            if trims:
                lines = [f"- {model} in stock:"]
                for item in trims:
                    lines.append(f"  • {item['trim']} – {item['color']} (ID: {item['stock_id']})")
                parts.append("\n".join(lines))
            else:
                parts.append(f"- {model}: currently out of stock.")
        body = "\n".join(parts)
        return (
            f"Here’s what we have right now:\n{body}\n\n"
            "If you want, I can place a hold or text you when new units arrive."
        )
    else:
        summary_lines = [f"- {model}: {len(items)} in stock" for model, items in INVENTORY.items()]
        summary = "\n".join(summary_lines)
        return (
            "We’ve got several popular models available right now:\n"
            f"{summary}\n\n"
            "Tell me which model/trim/color you’re interested in, and I can check exact availability."
        )


def respond_finance(_: str) -> str:
    return FINANCE_COPY


def respond_location(_: str) -> str:
    return (
        f"{DEALERSHIP_NAME}\n"
        f"Address: {DEALERSHIP_ADDRESS}\n"
        f"Phone: {DEALERSHIP_PHONE}\n"
        f"Hours: {DEALERSHIP_HOURS}\n\n"
        "We’re just off the freeway with plenty of parking. Would you like directions?"
    )


# ---------- Trade-In Appraisal Engine ---------------------------------------

@dataclass
class TradeInDraft:
    year: Optional[int] = None
    make: Optional[str] = None
    model: Optional[str] = None
    mileage: Optional[int] = None
    condition: Optional[str] = None
    zip_code: Optional[str] = None
    vin: Optional[str] = None
    accidents: Optional[bool] = None
    name: Optional[str] = None
    phone: Optional[str] = None


@dataclass
class TradeInEstimate:
    low: int
    high: int
    assumptions: str


class TradeInAppraiser:
    def estimate(self, *, year: int, make: str, model: str, mileage: int, condition: str, accidents: Optional[bool]) -> TradeInEstimate:
        now_year = datetime.now().year
        age = max(0, now_year - year)
        make_norm = (make or "").lower()

        # Segment baselines (very rough)
        segment_baseline = 35000
        if make_norm in {"bmw","mercedes","mercedes-benz","audi","lexus","infiniti","tesla","volvo"}:
            segment_baseline = 45000
        elif make_norm in {"kia","hyundai","chevrolet","chevy","ford","nissan","volkswagen","vw","subaru"}:
            segment_baseline = 32000
        elif make_norm in {"jeep","gmc","ram","dodge"}:
            segment_baseline = 38000

        value = float(segment_baseline)
        if age >= 1:
            value *= (1 - 0.18)
            for _ in range(age - 1):
                value *= (1 - 0.12)

        expected_miles = 12000 * age
        delta = mileage - expected_miles
        per_mile_penalty = 0.10
        per_mile_credit = 0.05
        if delta > 0:
            value -= per_mile_penalty * min(delta, 200000)
        else:
            value += per_mile_credit * min(abs(delta), 100000)

        cond = (condition or "good").lower()
        cond_mult = {"excellent": 1.05, "good": 1.00, "fair": 0.90, "poor": 0.75}.get(cond, 1.0)
        value *= cond_mult

        if accidents is True:
            value *= 0.90

        value = max(500.0, value)
        low = int(max(500, round(value * 0.92)))
        high = int(round(value * 1.08))

        assumptions = (
            f"Assumptions: age={age}y, expected_miles≈{expected_miles:,}, "
            f"condition='{cond}', accidents={'yes' if accidents else 'no/unknown'}."
        )
        return TradeInEstimate(low=low, high=high, assumptions=assumptions)


# ---------- Appointment Flow --------------------------------------------------

@dataclass
class AppointmentDraft:
    model: Optional[str] = None
    when: Optional[datetime] = None
    name: Optional[str] = None
    phone: Optional[str] = None


@dataclass
class ConversationState:
    last_service_intent: Optional[str] = None
    awaiting_satisfaction: bool = False
    history: List[Tuple[str, str]] = field(default_factory=list)
    collecting_appointment: bool = False
    appointment_draft: AppointmentDraft = field(default_factory=AppointmentDraft)
    collecting_tradein: bool = False
    tradein_draft: TradeInDraft = field(default_factory=TradeInDraft)


class CustomerServiceBot:
    def __init__(self, appt_store: Optional[AppointmentStore] = None, hist_store: Optional[HistoryStore] = None, appr_store: Optional[AppraisalStore] = None):
        self.state = ConversationState()
        self.appts = appt_store or AppointmentStore()
        self.history = hist_store or HistoryStore()
        self.appraisals = appr_store or AppraisalStore()
        self.appraiser = TradeInAppraiser()

    # ----- Main entry -----
    def handle_message(self, user_text: str) -> str:
        intent = detect_intent(user_text)

        # Active flows supersede (unless direct satisfaction)
        if self.state.collecting_appointment and intent not in ("affirmation","negation"):
            reply = self._appointment_collect(user_text)
            self._log(user_text, reply)
            return reply
        if self.state.collecting_tradein and intent not in ("affirmation","negation"):
            reply = self._tradein_collect(user_text)
            self._log(user_text, reply)
            return reply

        # Satisfaction
        if self.state.awaiting_satisfaction and intent in ("affirmation","negation"):
            self.state.awaiting_satisfaction = False
            if intent == "affirmation":
                reply = "Great! If there’s anything else you need—test drives, quotes, or trade-ins—just let me know."
                self._log(user_text, reply)
                return reply
            else:
                next_step = {
                    "inventory": "Which model/trim should I check specifically? I can also set a stock alert for you.",
                    "finance": "What monthly budget or term are you considering? I can draft a sample quote.",
                    "location": "Are you starting from a specific address? I can send step-by-step directions.",
                    "appointment": "Would you like a morning or afternoon slot? I can suggest a few times.",
                    "tradein": "I can refine your appraisal—do you know the VIN or have photos? We can also book a 15-minute in-person check.",
                }.get(self.state.last_service_intent, "Tell me what you’d like to do next and I’ll take care of it.")
                reply = f"Thanks for the feedback. Let’s get this right. {next_step}"
                self._log(user_text, reply)
                return reply

        # Route main intents
        if intent == "inventory":
            answer = respond_inventory(user_text)
            reply = self._with_satisfaction(answer, "inventory")
        elif intent == "finance":
            answer = respond_finance(user_text)
            reply = self._with_satisfaction(answer, "finance")
        elif intent == "location":
            answer = respond_location(user_text)
            reply = self._with_satisfaction(answer, "location")
        elif intent == "appointment":
            reply = self._appointment_start(user_text)
        elif intent == "tradein":
            reply = self._tradein_start(user_text)
        elif intent in ("affirmation","negation"):
            reply = "Awesome! Anything else I can help with today?" if intent == "affirmation" else "No problem—what info would you like me to clarify or provide next?"
        else:
            reply = (
                "I can help with inventory availability, finance/lease options, our location/hours, "
                "schedule a test-drive, or give you an instant trade-in appraisal.\n"
                "What would you like to do?"
            )

        self._log(user_text, reply)
        return reply

    # ----- Shared helpers -----
    def _with_satisfaction(self, message: str, intent_name: str) -> str:
        self.state.last_service_intent = intent_name
        self.state.awaiting_satisfaction = True
        return f"{message}\n\nDid that answer your question? (yes/no)"

    def _log(self, user_text: str, bot_text: str) -> None:
        self.state.history.append((user_text, bot_text))
        self.history.append(user_text, bot_text)

    # ----- Appointment Flow -----
    def _appointment_start(self, user_text: str) -> str:
        self.state.last_service_intent = "appointment"
        self.state.collecting_appointment = True
        models = extract_models_from_text(user_text)
        if models:
            self.state.appointment_draft.model = models[0]
        when = parse_datetime_natural(user_text)
        if when:
            self.state.appointment_draft.when = when
        return self._appointment_collect(user_text)

    def _appointment_collect(self, user_text: str) -> str:
        d = self.state.appointment_draft
        if not d.model:
            models = extract_models_from_text(user_text)
            if models:
                d.model = models[0]
        if not d.when:
            pdt = parse_datetime_natural(user_text)
            if pdt:
                d.when = pdt
        if not d.name:
            m = re.search(r"\bmy name is ([a-z ,.'-]+)", user_text, re.I)
            if m:
                d.name = m.group(1).strip().title()
        if not d.phone:
            m = re.search(r"(\+?1?[-.\s]?)?\(?\d{3}\)?[-.\s]?\d{3}[-.\s]?\d{4}\b", user_text)
            if m:
                d.phone = m.group(0)

        if not d.model:
            available = ", ".join(INVENTORY.keys())
            return f"Happy to set that up! Which model would you like to test drive? (e.g., {available})"
        if not d.when:
            return (
                "Great choice. What date and time works for you? "
                "You can say things like '2025-10-12 14:30', '10/12 2:30 pm', or 'tomorrow 3pm'."
            )

        if not self._is_within_hours(d.when):
            return (
                "We’re open Mon–Sat 9:00am–7:00pm and Sun 10:00am–5:00pm. "
                "Please pick a time within business hours."
            )

        if self.appts.is_conflict(d.when):
            alt1 = d.when + timedelta(minutes=90)
            alt2 = d.when + timedelta(minutes=150)
            return (
                f"That time is booked. Would {alt1.strftime('%a %b %d, %I:%M %p')} or "
                f"{alt2.strftime('%a %b %d, %I:%M %p')} work instead?"
            )

        appt = Appointment(model=d.model, when=d.when.isoformat(), name=d.name, phone=d.phone)
        self.appts.add(appt)

        # Notify (stubs)
        send_sms_stub(d.phone, f"Confirmed: {DEALERSHIP_NAME} test drive for {d.model} on {datetime.fromisoformat(appt.when).strftime('%a %b %d at %I:%M %p')}")

        # Reset and ask satisfaction
        self.state.collecting_appointment = False
        self.state.appointment_draft = AppointmentDraft()
        self.state.awaiting_satisfaction = True
        self.state.last_service_intent = "appointment"

        contact_line = ""
        if appt.name or appt.phone:
            pieces = []
            if appt.name: pieces.append(appt.name)
            if appt.phone: pieces.append(appt.phone)
            contact_line = f"\nContact: {', '.join(pieces)}"

        return (
            f"All set! I’ve scheduled your {appt.model} test drive for "
            f"{datetime.fromisoformat(appt.when).strftime('%A, %B %d at %I:%M %p')}.{contact_line}\n"
            "Please bring your driver’s license.\n\nDid that answer your request? (yes/no)"
        )

    def _is_within_hours(self, when: datetime) -> bool:
        weekday = when.weekday()  # Mon=0, Sun=6
        hour = when.hour + when.minute/60.0
        if weekday == 6:  # Sunday
            return 10 <= hour < 17
        return 9 <= hour < 19

    # ----- Trade-in Flow -----
    def _tradein_start(self, user_text: str) -> str:
        self.state.last_service_intent = "tradein"
        self.state.collecting_tradein = True
        self._tradein_autofill(user_text)
        return self._tradein_collect(user_text)

    def _tradein_autofill(self, text: str) -> None:
        d = self.state.tradein_draft
        t = text.lower()

        if d.year is None:
            m = re.search(r"\b(20\d{2}|19\d{2})\b", t)
            if m:
                d.year = int(m.group(1))
        if d.mileage is None:
            m = re.search(r"\b(\d{1,3}[,\s]?\d{3}|\d{1,6})\s*miles?\b", t)
            if m:
                d.mileage = int(re.sub(r"[^\d]", "", m.group(1)))
        if d.zip_code is None:
            m = re.search(r"\b(\d{5})(?:-\d{4})?\b", t)
            if m:
                d.zip_code = m.group(1)
        if d.vin is None:
            m = re.search(r"\b([0-9a-hj-npr-z]{17})\b", t)
            if m:
                d.vin = m.group(1).upper()
        if d.accidents is None:
            if "no accident" in t or "clean title" in t:
                d.accidents = False
            elif "accident" in t or "salvage" in t:
                d.accidents = True
        if d.condition is None:
            for lvl in CONDITION_LEVELS:
                if lvl in t:
                    d.condition = lvl
                    break
        if d.make is None or d.model is None:
            tokens = re.findall(r"[a-z0-9\-]+", t)
            for i, tok in enumerate(tokens):
                if tok in KNOWN_MAKES and i + 1 < len(tokens):
                    d.make = tokens[i]
                    d.model = tokens[i + 1]
                    break
        if d.name is None:
            m = re.search(r"\bmy name is ([a-z ,.'-]+)", text, re.I)
            if m:
                d.name = m.group(1).strip().title()
        if d.phone is None:
            m = re.search(r"(\+?1?[-.\s]?)?\(?\d{3}\)?[-.\s]?\d{3}[-.\s]?\d{4}\b", text)
            if m:
                d.phone = m.group(0)

    def _tradein_collect(self, user_text: str) -> str:
        d = self.state.tradein_draft
        self._tradein_autofill(user_text)

        if d.year is None:
            return "Let’s get your instant appraisal. What **year** is your vehicle? (e.g., 2018)"
        if d.make is None:
            return "Great—what **make** is it? (e.g., Honda, Toyota, Ford)"
        if d.model is None:
            return "And what **model** is it? (e.g., Civic, RAV4, F-150)"
        if d.mileage is None:
            return "Approximately how many **miles** are on it?"
        if d.condition is None:
            return "How would you rate the **condition**? (excellent / good / fair / poor)"
        if d.accidents is None:
            return "Any **accidents** or reported damage? (yes/no)"
        if d.zip_code is None:
            return "What’s your **ZIP code**? (optional, helps us tailor local demand) Or say 'skip'."
        if d.vin is None:
            return "If you have the **VIN**, share it (optional). Or say 'skip'."

        est = self.appraiser.estimate(year=d.year, make=d.make, model=d.model, mileage=d.mileage, condition=d.condition, accidents=d.accidents)

        appr = Appraisal(
            created_at=datetime.utcnow().isoformat(),
            year=d.year, make=d.make, model=d.model, mileage=d.mileage,
            condition=d.condition or "good", accidents=d.accidents,
            zip_code=d.zip_code, vin=d.vin, name=d.name, phone=d.phone,
            low=est.low, high=est.high, assumptions=est.assumptions,
        )
        self.appraisals.add(appr)

        # Notify (stubs)
        send_sms_stub(d.phone, f"Your {d.year} {d.make} {d.model} trade-in estimate: ${est.low:,}-${est.high:,} (subject to inspection)")

        self.state.collecting_tradein = False
        self.state.awaiting_satisfaction = True
        self.state.last_service_intent = "tradein"
        self.state.tradein_draft = TradeInDraft()

        contact_line = []
        if appr.name: contact_line.append(appr.name)
        if appr.phone: contact_line.append(appr.phone)
        contact_line = f"\nContact: {', '.join(contact_line)}" if contact_line else ""

        return (
            "Here’s your **instant trade-in estimate** (not a final offer):\n"
            f"• Estimated range: ${est.low:,} – ${est.high:,}\n"
            f"• Vehicle: {appr.year} {appr.make.title()} {appr.model.title()} | {appr.mileage:,} miles | condition: {appr.condition}\n"
            f"• {est.assumptions}\n"
            f"{contact_line}\n\n"
            "Next steps: we can **lock in a firmer offer** after a quick in-person check (10–15 minutes). "
            "Would you like to book a visit or apply this toward a new vehicle?\n\n"
            "Did that answer your request? (yes/no)"
        )


# ---------- CLI --------------------------------------------------------------

def run_cli():
    print(f"Welcome to {DEALERSHIP_NAME} virtual assistant! Type 'exit' to quit.\n")
    bot = CustomerServiceBot()
    while True:
        try:
            user = input("You: ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\nGoodbye!")
            break
        if user.lower() in ("exit", "quit"):
            print("Bot: Thanks for visiting. Have a great day!")
            break
        response = bot.handle_message(user)
        print(f"Bot: {response}\n")


# ---------- Admin Commands ---------------------------------------------------

def admin_cmd(args: argparse.Namespace):
    appts = AppointmentStore()
    hist = HistoryStore()
    apprs = AppraisalStore()

    if args.admin == "list-appointments":
        for a in appts.list_all():
            print(f"{a.when} — {a.model} — {a.name or ''} {a.phone or ''}")

    elif args.admin == "cancel-appointment":
        if not args.at:
            raise SystemExit("--at ISO_DATETIME required (e.g., 2025-10-12T14:30:00)")
        ok = appts.cancel_at(args.at)
        print("Cancelled." if ok else "No appointment matched that time.")

    elif args.admin == "export-appointments":
        if not args.csv:
            raise SystemExit("--csv path required")
        appts.export_csv(args.csv)
        print(f"Exported to {args.csv}")

    elif args.admin == "export-history":
        if not args.csv:
            raise SystemExit("--csv path required")
        hist.export_csv(args.csv)
        print(f"Exported to {args.csv}")

    elif args.admin == "export-appraisals":
        if not args.csv:
            raise SystemExit("--csv path required")
        apprs.export_csv(args.csv)
        print(f"Exported to {args.csv}")

    else:
        raise SystemExit("Unknown admin command")


# ---------- API (FastAPI) ----------------------------------------------------

def build_api():
    try:
        from fastapi import FastAPI
        from pydantic import BaseModel
    except Exception as e:
        raise SystemExit("FastAPI/Pydantic not installed. Run: pip install fastapi uvicorn pydantic")

    app = FastAPI(title="Dealership Assistant API", version="1.0")
    bot = CustomerServiceBot()

    class ChatIn(BaseModel):
        message: str

    class ChatOut(BaseModel):
        reply: str

    @app.post("/chat", response_model=ChatOut)
    def chat(inp: ChatIn):
        reply = bot.handle_message(inp.message)
        return ChatOut(reply=reply)

    class AppointmentIn(BaseModel):
        model: str
        when: str  # ISO datetime
        name: Optional[str] = None
        phone: Optional[str] = None

    class AppointmentOut(BaseModel):
        ok: bool
        detail: str

    @app.post("/appointments", response_model=AppointmentOut)
    def create_appointment(a: AppointmentIn):
        when_dt = datetime.fromisoformat(a.when)
        if not bot._is_within_hours(when_dt):
            return AppointmentOut(ok=False, detail="Outside business hours")
        if bot.appts.is_conflict(when_dt):
            return AppointmentOut(ok=False, detail="Time slot unavailable")
        appt = Appointment(model=a.model, when=when_dt.isoformat(), name=a.name, phone=a.phone)
        bot.appts.add(appt)
        send_sms_stub(a.phone, f"Confirmed: {DEALERSHIP_NAME} test drive for {a.model} on {when_dt.strftime('%a %b %d at %I:%M %p')}")
        return AppointmentOut(ok=True, detail="Appointment created")

    @app.get("/appointments")
    def list_appointments():
        return [asdict(a) for a in bot.appts.list_all()]

    @app.delete("/appointments")
    def cancel_appointment(at: str):
        ok = bot.appts.cancel_at(at)
        return {"ok": ok}

    class AppraisalIn(BaseModel):
        year: int
        make: str
        model: str
        mileage: int
        condition: str
        accidents: Optional[bool] = None
        zip_code: Optional[str] = None
        vin: Optional[str] = None
        name: Optional[str] = None
        phone: Optional[str] = None

    @app.post("/appraisals")
    def create_appraisal(a: AppraisalIn):
        est = bot.appraiser.estimate(year=a.year, make=a.make, model=a.model, mileage=a.mileage, condition=a.condition, accidents=a.accidents)
        appr = Appraisal(
            created_at=datetime.utcnow().isoformat(),
            year=a.year, make=a.make, model=a.model, mileage=a.mileage,
            condition=a.condition, accidents=a.accidents,
            zip_code=a.zip_code, vin=a.vin, name=a.name, phone=a.phone,
            low=est.low, high=est.high, assumptions=est.assumptions,
        )
        bot.appraisals.add(appr)
        send_sms_stub(a.phone, f"Your {a.year} {a.make} {a.model} trade-in estimate: ${est.low:,}-${est.high:,} (subject to inspection)")
        return asdict(appr)

    @app.get("/appraisals")
    def list_appraisals():
        return [asdict(a) for a in bot.appraisals.list_all()]

    @app.get("/health")
    def health():
        return {"status": "ok"}

    return app


# ---------- Unit Tests -------------------------------------------------------

def run_tests():
    import unittest

    class TestNLP(unittest.TestCase):
        def test_tradein_intent(self):
            self.assertEqual(detect_intent("Can I get a trade-in appraisal?"), "tradein")
        def test_appt_intent(self):
            self.assertEqual(detect_intent("I want to schedule a test drive"), "appointment")
        def test_inventory_intent(self):
            self.assertEqual(detect_intent("Do you have any CR-V in stock?"), "inventory")

    class TestHours(unittest.TestCase):
        def test_within_hours_weekday(self):
            bot = CustomerServiceBot()
            dt = datetime(2025, 10, 7, 10, 0)  # Tue 10:00
            self.assertTrue(bot._is_within_hours(dt))
        def test_outside_hours_sun_night(self):
            bot = CustomerServiceBot()
            dt = datetime(2025, 10, 5, 20, 0)  # Sun 8pm
            self.assertFalse(bot._is_within_hours(dt))

    class TestAppraisalMath(unittest.TestCase):
        def test_estimate_nonzero(self):
            app = TradeInAppraiser()
            est = app.estimate(year=2018, make="Honda", model="Civic", mileage=65000, condition="good", accidents=False)
            self.assertTrue(est.low >= 500)
            self.assertTrue(est.high > est.low)

    suite = unittest.defaultTestLoader.loadTestsFromModule(__import__(__name__))
    runner = unittest.TextTestRunner(verbosity=2)
    result = runner.run(suite)
    return 0 if result.wasSuccessful() else 1


# ---------- Entrypoint -------------------------------------------------------

def main():
    parser = argparse.ArgumentParser(description="Dealership Assistant")
    parser.add_argument("--cli", action="store_true", help="Run interactive CLI chat")
    parser.add_argument("--api", action="store_true", help="Run FastAPI server")
    parser.add_argument("--host", default="127.0.0.1")
    parser.add_argument("--port", type=int, default=8000)

    # Admin subcommands
    parser.add_argument("--admin", choices=[
        "list-appointments","cancel-appointment","export-appointments","export-history","export-appraisals"
    ])
    parser.add_argument("--at", help="ISO datetime for cancel-appointment (e.g., 2025-10-12T14:30:00)")
    parser.add_argument("--csv", help="Path for CSV export")

    parser.add_argument("--test", action="store_true", help="Run unit tests")

    # Make admin optional/independent flags
    parser.set_defaults(admin=None)

    args = parser.parse_args()

    if args.test:
        raise SystemExit(run_tests())

    if args.admin:
        return admin_cmd(args)

    if args.api:
        app = build_api()
        try:
            import uvicorn
        except Exception:
            raise SystemExit("Uvicorn not installed. Run: pip install uvicorn")
        uvicorn.run(app, host=args.host, port=args.port)
        return

    # Default to CLI if explicitly requested or nothing else chosen
    if args.cli or (not args.api and not args.admin):
        run_cli()
        return


if __name__ == "__main__":
    main()